# Laboratorio reproducible: algoritmos de clustering

Este cuaderno acompaña la investigación teórica del README. Compara **K-Means**, **clustering jerárquico aglomerativo**, **DBSCAN** y **Gaussian Mixture Models (GMM)** sobre datos sin etiquetas.

## Objetivos

1. Visualizar cómo cambia el resultado según la hipótesis geométrica de cada algoritmo.
2. Comprobar por qué DBSCAN detecta formas no convexas y ruido.
3. Usar el método del codo y el coeficiente de silueta para elegir un valor razonable de $K$ en K-Means.

> **Reproducibilidad:** todas las semillas aleatorias están fijadas con `random_state=42`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.datasets import make_moons
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Datos sin etiquetas con geometría no convexa

Los dos semicírculos representan una situación en la que la proximidad al centro no coincide con la estructura real. Es un buen caso para contrastar métodos particionales, jerárquicos y basados en densidad.

In [ ]:
X, _ = make_moons(n_samples=600, noise=0.07, random_state=RANDOM_STATE)
X = StandardScaler().fit_transform(X)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X[:, 0], X[:, 1], s=20, color='#38bdf8', edgecolor='white', linewidth=0.25)
ax.set(title='Datos de partida: dos lunas sin etiquetas', xlabel='Característica 1', ylabel='Característica 2')
plt.show()

## 2. Comparación de algoritmos

Para facilitar una comparación justa, los métodos que necesitan el número de grupos reciben $K=2$. DBSCAN no recibe $K$: usa `eps` y `min_samples` para definir densidad.

In [ ]:
models = {
    'K-Means ($K=2$)': KMeans(n_clusters=2, n_init=20, random_state=RANDOM_STATE),
    'Jerárquico aglomerativo': AgglomerativeClustering(n_clusters=2, linkage='ward'),
    'DBSCAN ($\varepsilon=0.22$, MinPts=8)': DBSCAN(eps=0.22, min_samples=8),
    'GMM (2 componentes)': GaussianMixture(n_components=2, random_state=RANDOM_STATE),
}

labels = {}
for name, model in models.items():
    labels[name] = model.fit_predict(X)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), constrained_layout=True)
for ax, (name, group_labels) in zip(axes, labels.items()):
    ax.scatter(X[:, 0], X[:, 1], c=group_labels, cmap='viridis', s=18, edgecolor='white', linewidth=0.2)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('La hipótesis del algoritmo condiciona el agrupamiento', fontsize=15, fontweight='bold')
plt.show()

### Lectura del resultado

- **K-Means** y **GMM** separan el plano a partir de centros/componentes; por ello tienden a cortar las lunas.
- El método **jerárquico** depende de la métrica y del enlace elegido; `ward` favorece grupos compactos.
- **DBSCAN** recupera los dos arcos porque expande regiones de alta densidad conectadas. Las etiquetas `-1`, si aparecen, representan ruido.

## 3. Elegir $K$: codo y silueta

Los dos criterios se calculan para K-Means con varios valores de $K$. El codo busca cuándo la reducción de inercia deja de compensar; la silueta mide cohesión interna frente a separación entre grupos.

In [ ]:
k_values = range(2, 9)
inertias, silhouettes = [], []

for k in k_values:
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    prediction = model.fit_predict(X)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X, prediction))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(k_values, inertias, marker='o', color='#2563eb', linewidth=2)
axes[0].set(title='Método del codo', xlabel='Número de clusters ($K$)', ylabel='Inercia')

axes[1].plot(k_values, silhouettes, marker='o', color='#0f766e', linewidth=2)
axes[1].set(title='Coeficiente de silueta', xlabel='Número de clusters ($K$)', ylabel='Silueta media')

plt.show()

best_k = list(k_values)[int(np.argmax(silhouettes))]
print(f'Mejor silueta en este conjunto para K = {best_k}: {max(silhouettes):.3f}')

## Conclusión

No existe un algoritmo universalmente mejor. La decisión debe combinar la geometría de los datos, la presencia de ruido, la interpretación de las variables y métricas de validación. En estas lunas, el resultado visual de DBSCAN es más fiel a la estructura generadora; eso no implica que siempre sea la mejor alternativa para otros datos o escalas.